# Colab Actions Workbench

Bu notebook, GitHub Actions workflow dosyalarını `.github/workflows/` altından otomatik keşfeder ve desteklediği adımları **aktif Google Colab Linux runtime'ında** yürütür. GitHub Actions çalıştırması başlatmaz; GitHub Actions kotasını kullanmaz.

## Başlangıç

1. Notebook'u Google Colab'da aç ve başlatma hücresini çalıştır.
2. Public repository için token gerekmez. Private repository veya GitHub depo/branch listeleme için Colab **Secrets** bölümüne `GH_TOKEN` adlı secret ekle.
3. Repository ve ref alanlarını doğrula, **Depoyu indir ve workflow'ları tara** düğmesine bas.
4. Workflow ve event'i seç, önce planı analiz et. Komutları gözden geçirdikten sonra güven onayını işaretleyerek çalıştır.
5. JSON/Markdown raporlarını indir. Colab oturumu bittiğinde runtime dosyaları silinebilir.

Varsayılan repository/ref Tessera'nın bu notebook'un bulunduğu geliştirme dalıdır; arayüzden başka bir repository ve branch/tag/commit seçilebilir.

## Ne yapar?

- `.github/workflows` altındaki YAML dosyalarını tarar; `workflow_dispatch` input alanlarını form olarak üretir.
- Job bağımlılıklarını (`needs`), basit matrix kombinasyonlarını, temel `if` ifadelerini ve GitHub expression alanlarını işler.
- Linux uyumlu `bash`, `sh` ve Python komutlarını Colab'da çalıştırır.
- `actions/checkout`, yerel composite action, yerel upload/download artifact ve bazı yaygın action'lar için sınırlı uyumluluk sağlar.
- Timeout, iptal isteği, adım logları ve JSON/Markdown sonuç raporu üretir.

## Önemli sınırlar

Colab Linux, Windows/macOS GitHub runner değildir. Windows/MSVC, PowerShell, Microsoft Detours, Windows loader/unwind davranışı ve Windows'a özel fuzz işlemleri burada çalıştırılmaz. Bunlar açıkça **BLOCKED_PLATFORM** veya **UNSUPPORTED** raporlanır; PASS sayılmaz. Marketplace action'larının çoğu, Docker/Node tabanlı özel action'lar, reusable workflow'lar, servis container'ları, GitHub'a check/status/release yayınlama ve tüm Actions ifadeleri birebir emüle edilmez. Ayrıntı: [Colab Actions Workbench dokümantasyonu](https://github.com/Jue-dev/Tessera/blob/issue-6/unwind-lifecycle/docs/colab-actions-workbench.md).

## Güvenlik

Workflow YAML'ları ve içlerindeki shell komutları kod çalıştırır. Colab bir güvenlik sandbox'ı değildir. **Yalnızca güvendiğin repository ve branch'lerde çalıştır.** Token'ı notebook içine yazma; Colab Secrets kullan. Token clone/API işlemleri için kullanılır, workflow step ortamına otomatik aktarılmaz. `secrets.*` ifadeleri bilerek bloke edilir.

In [ ]:
%pip -q install pyyaml ipywidgets

import pathlib
import py_compile
import runpy
import urllib.request

# Runner engine source. Update ENGINE_REF when this notebook is moved to another branch.
ENGINE_REPO = "Jue-dev/Tessera"
ENGINE_REF = "issue-6/unwind-lifecycle"
ENGINE_PATH = "verify/colab_actions/colab_actions_runner.py"

engine_url = (
    f"https://raw.githubusercontent.com/{ENGINE_REPO}/"
    f"{ENGINE_REF}/{ENGINE_PATH}"
)
engine_file = pathlib.Path("/content/colab_actions_runner.py")

request = urllib.request.Request(
    engine_url,
    headers={"User-Agent": "Tessera-Colab-Actions-Workbench"},
)
with urllib.request.urlopen(request, timeout=45) as response:
    engine_source = response.read()

if len(engine_source) < 5000:
    raise RuntimeError(
        "Runner engine response is unexpectedly small. "
        "Check ENGINE_REPO, ENGINE_REF and ENGINE_PATH."
    )

engine_file.write_bytes(engine_source)
py_compile.compile(str(engine_file), doraise=True)
print(f"Runner engine loaded and syntax-checked: {engine_file}")
runpy.run_path(str(engine_file), run_name="__main__")
